In [1]:
import pandas as pd
df = pd.read_csv("../data/Coffee.csv")
df.head()

,transaction_id,year,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2025,7:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2025,7:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2025,7:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2025,7:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2025,7:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [2]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   year              149116 non-null  int64  
 2   transaction_time  149116 non-null  object 
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  object 
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  object 
 9   product_type      149116 non-null  object 
 10  product_detail    149116 non-null  object 
dtypes: float64(1), int64(5), object(5)
memory usage: 12.5+ MB


transaction_id      0
year                0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64

In [3]:
df = df.dropna()
df = df[df["transaction_qty"] > 0]
df = df[df["unit_price"] > 0]

In [4]:
df["revenue"] = df["transaction_qty"] * df["unit_price"]

In [6]:
product_volume = df.groupby("product_detail")["transaction_qty"].sum().reset_index()

product_volume = product_volume.sort_values(by="transaction_qty",ascending=False)

In [14]:
top10_volume = product_volume.head(10)

In [12]:
product_revenue = df.groupby("product_detail")["revenue"].sum().reset_index()

product_revenue = product_revenue.sort_values(by="revenue",ascending=False)

In [15]:
total_revenue = df["revenue"].sum()

product_revenue["revenue_share"] = (product_revenue["revenue"] / total_revenue) * 100

In [16]:
comparison = pd.merge(product_volume,product_revenue,on="product_detail")

comparison["volume_rank"] = comparison["transaction_qty"].rank(ascending=False)

comparison["revenue_rank"] = comparison["revenue_rank"] = comparison["revenue"].rank(ascending=False)

In [17]:
category_rev = df.groupby("product_category")["revenue"].sum().reset_index()

category_rev["share"] = (category_rev["revenue"] / total_revenue) * 100

In [18]:
type_rev = df.groupby("product_type")["revenue"].sum().reset_index()

type_rev = type_rev.sort_values(by="revenue",ascending=False)

In [19]:
pareto = product_revenue.copy()

pareto["cum_revenue"] = pareto["revenue"].cumsum()

pareto["cum_percent"] = (pareto["cum_revenue"] / total_revenue) * 100

In [20]:
hero_products = pareto[pareto["cum_percent"] <= 80]

In [21]:
long_tail = pareto[pareto["revenue_share"]< 1]

In [22]:
top20_products = int(len(pareto) * 0.2)

concentration_ratio = (pareto.head(top20_products)["revenue"].sum() / total_revenue) * 100

In [23]:
efficiency = df.groupby("product_detail")["revenue"].sum().reset_index()

efficiency["efficiency_score"] = efficiency["revenue"]